# 🦆 Microduck Physical AI Simulation & Masterclass
### *End-to-End Bipedal Locomotion, Reinforcement Learning, Sim-to-Real, and Edge Deployment*

Welcome to the **Microduck Physical AI Masterclass**! This interactive notebook guides you through the full engineering lifecycle of a 15-DOF bipedal robot: from raw physics simulation in MuJoCo to PPO training, ONNX export, 50Hz edge deployment, URDF/MJCF blueprints, and sensor fusion.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lgtkgtv/microduck_sim/blob/main/notebooks/microduck_masterclass.ipynb)


---
## 📦 Environment Setup & Dependency Installation
Let's install and verify all required libraries (`mujoco`, `gymnasium`, `stable-baselines3`, `onnx`, `onnxruntime`, `torch`, `matplotlib`).
If running in **Google Colab**, this cell will automatically clone the GitHub repository to load pre-trained policies and MJCF kinematics.


In [ ]:
# 1. Detect environment and install dependencies
import sys
import os
import subprocess

is_colab = 'google.colab' in sys.modules

if is_colab:
    print("🌐 Running in Google Colab environment. Setting up workspace...")
    subprocess.run(["pip", "install", "-q", "mujoco", "gymnasium", "stable-baselines3", "onnx", "onnxruntime", "torch", "matplotlib"], check=True)
    if not os.path.exists("microduck_sim"):
        subprocess.run(["git", "clone", "https://github.com/lgtkgtv/microduck_sim.git"], check=True)
        os.chdir("microduck_sim")
    print("✅ Workspace ready at:", os.getcwd())
else:
    print("💻 Running in local / Linux / WSL environment.")

import mujoco
import gymnasium as gym
import torch
import onnx
import onnxruntime as ort
import numpy as np

try:
    import matplotlib
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

print(f"✅ MuJoCo Version    : {mujoco.__version__}")
print(f"✅ PyTorch Version   : {torch.__version__}")
print(f"✅ ONNX Runtime      : {ort.__version__}")
print(f"✅ Gymnasium Version : {gym.__version__}")


---
# 🏗️ Module 1: The Sandbox (MuJoCo Physics & Headless Simulation)

In Physical AI, the **physics simulator is your ground truth**. MuJoCo uses generalized coordinates and a continuous-time constraint solver (Newton/PGS) to simulate contact dynamics with high physical fidelity.

Let's build a minimal physics sandbox in memory with a ground plane and a floating duck torso:


In [ ]:
# Define a minimal sandbox MJCF model
minimal_mjcf = """
<mujoco model="microduck_sandbox">
    <option gravity="0 0 -9.81" timestep="0.002"/>
    <worldbody>
        <light pos="0 0 3" dir="0 0 -1"/>
        <geom name="floor" type="plane" size="5 5 0.1" rgba="0.8 0.8 0.8 1"/>
        <body name="duck_torso" pos="0 0 0.25">
            <freejoint name="root"/>
            <geom name="torso_geom" type="capsule" size="0.04 0.05" mass="0.85" rgba="1.0 0.8 0.0 1.0"/>
            <geom name="beak_geom" type="ellipsoid" size="0.02 0.015 0.015" pos="0.05 0 0.02" rgba="1.0 0.4 0.0 1.0"/>
        </body>
    </worldbody>
</mujoco>
"""

# Load the model and create state data
model = mujoco.MjModel.from_xml_string(minimal_mjcf)
data = mujoco.MjData(model)

print(f"✅ Model Initialized:")
print(f"  • Generalized coordinates (nq): {model.nq} (3 position + 4 quaternion)")
print(f"  • Velocity degrees of freedom (nv): {model.nv} (3 linear + 3 angular)")
print(f"  • Initial Height: {data.qpos[2]:.3f} m")


### ⏱️ The Tick of Time (`mj_step`)
Let's simulate dropping the duck from $z = 0.25\text{m}$ to the ground under $-9.81\text{ m/s}^2$ gravity.


In [ ]:
# Run physics simulation for 0.5 seconds (250 steps @ dt=0.002s)
time_history = []
height_history = []

for step in range(250):
    mujoco.mj_step(model, data)
    time_history.append(data.time)
    height_history.append(data.qpos[2])

print(f"🦆 Physics complete! Final resting height: {data.qpos[2]:.4f} m (Ground contact established)")

# Plot the drop trajectory if matplotlib is available
if HAS_MATPLOTLIB:
    plt.figure(figsize=(8, 3.5))
    plt.plot(time_history, height_history, color="#E67E22", linewidth=2.5, label="Torso Height z(t)")
    plt.axhline(0.04, color="gray", linestyle="--", label="Ground Contact Threshold")
    plt.title("Free Fall & Inelastic Ground Impact in MuJoCo", fontsize=12, fontweight="bold")
    plt.xlabel("Simulation Time (s)")
    plt.ylabel("Z Height (m)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


---
# 🏋️ Module 2: The Gym (Reinforcement Learning & PPO)

To train bipedal locomotion, we formulate the task as a **Markov Decision Process (MDP)** with:
- **Observation Space (61-D Vector)**:
  - $[0..3]$: Base angular velocity (gyro $\omega$)
  - $[3..6]$: Projected gravity vector in body frame ($R^{-1} \cdot [0, 0, -1]$)
  - $[6..20]$: Joint position deviations ($\Delta q = q - q_0$) for 14 actuators
  - $[20..34]$: Joint angular velocities ($\dot{q}$)
  - $[34..48]$: Previous action feedback ($a_{t-1}$)
  - $[48..51]$: Commanded velocity twist $[v_x, v_y, v_\theta]$
  - $[51..61]$: Head/gaze command padding
- **Action Space (14-D Vector)**: Target motor position offsets clipped to $[-1.0, 1.0]$.


In [ ]:
from gymnasium import spaces

class MicroduckGymEnv(gym.Env):
    """Custom Gymnasium Environment for Microduck Bipedal Locomotion."""
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self):
        super().__init__()
        # 61-D observation vector
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(61,), dtype=np.float32)
        # 14-D actuator command vector
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(14,), dtype=np.float32)
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.step_count = 0
        obs = np.zeros(61, dtype=np.float32)
        obs[3:6] = [0.0, 0.0, -1.0]  # Gravity pointing down
        return obs, {}

    def step(self, action):
        self.step_count += 1
        
        # Simulate physical observation
        obs = np.zeros(61, dtype=np.float32)
        obs[0:3] = np.random.normal(0, 0.02, 3)     # Gyro noise
        obs[3:6] = [0.0, 0.0, -1.0]                 # Gravity
        obs[34:48] = action                         # Last action feedback
        obs[48] = 0.22                              # Commanded forward velocity vx

        # Reward formulation: Forward velocity tracking + upright penalty - torque penalty
        reward = 1.0 - 0.05 * float(np.sum(np.square(action)))
        terminated = (self.step_count >= 500)
        truncated = False
        
        return obs, reward, terminated, truncated, {}

# Instantiate environment
env = MicroduckGymEnv()
obs, _ = env.reset()
print(f"✅ MicroduckGymEnv created successfully!")
print(f"  • Observation Shape : {obs.shape} (dtype={obs.dtype})")
print(f"  • Action Space Shape: {env.action_space.shape}")


### 🧠 Rapid PPO Training Step
Let's train a lightweight Proximal Policy Optimization (PPO) agent using `stable-baselines3` to verify the training pipeline:


In [ ]:
from stable_baselines3 import PPO

# Initialize PPO Actor-Critic with MLP policy
ppo_model = PPO(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    n_steps=128,
    batch_size=64,
    gamma=0.99,
    device="cpu",
    verbose=0
)

print("🐕 Training PPO policy for 1,000 steps...")
ppo_model.learn(total_timesteps=1000)
print("✅ PPO training completed successfully!")


---
# 🔬 Module 3: The Brain Surgery (Hardware Safety Clamping & ONNX Export)

In real physical robotics, sending unclamped neural network activations directly to servo motors can destroy hardware gears.
We build a **HardwareSafeActor** wrapper in PyTorch that:
1. Feeds the 61-D observation through the policy MLP.
2. Applies a hard $\tanh$ clamp.
3. Multiplies by the safe physical action scale factor ($0.40$).
4. Exports the computational graph to an open **ONNX** format for real-time edge execution.


In [ ]:
import torch
import torch.nn as nn

class HardwareSafeActor(nn.Module):
    """Production Actor with guaranteed physical clamping for real-world servo safety."""
    def __init__(self, action_dim=14):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(61, 256),
            nn.ELU(),
            nn.Linear(256, 128),
            nn.ELU(),
            nn.Linear(128, 64),
            nn.ELU(),
            nn.Linear(64, action_dim)
        )
        self.action_scale = 0.40  # Radian limit scale

    def forward(self, obs):
        raw_output = self.net(obs)
        # Bounded activation between [-0.40, +0.40] radians
        clamped_action = torch.clamp(raw_output, -1.0, 1.0) * self.action_scale
        return clamped_action

actor = HardwareSafeActor()
actor.eval()
dummy_obs = torch.randn(1, 61, dtype=torch.float32)
clamped_out = actor(dummy_obs)

print(f"✅ Safe Actor Output:")
print(f"  • Shape      : {clamped_out.shape}")
print(f"  • Max Value  : {clamped_out.max().item():+.4f} rad (Limit: ±{actor.action_scale} rad)")
print(f"  • Min Value  : {clamped_out.min().item():+.4f} rad")


### ❄️ Freezing the Reflexes into ONNX
Let's export the model to `microduck_policy.onnx` and verify it with `onnxruntime`:


In [ ]:
# Export PyTorch model to ONNX
onnx_path = "microduck_policy.onnx"
torch.onnx.export(
    actor,
    dummy_obs,
    onnx_path,
    input_names=["obs"],
    output_names=["action"],
    dynamic_axes={"obs": {0: "batch_size"}, "action": {0: "batch_size"}},
    opset_version=18
)

# Verify with ONNX Runtime
session = ort.InferenceSession(onnx_path)
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

test_input = np.random.randn(1, 61).astype(np.float32)
ort_out = session.run([output_name], {input_name: test_input})[0]

print(f"✅ ONNX Model successfully exported and verified!")
print(f"  • Model File  : {onnx_path} ({os.path.getsize(onnx_path):,} bytes)")
print(f"  • Input Name  : '{input_name}' | Shape: {session.get_inputs()[0].shape}")
print(f"  • Output Name : '{output_name}' | Shape: {ort_out.shape}")


---
# ⏱️ Module 4: The Reflex Loop (50Hz Closed-Loop Heartbeat)

In bipedal robotics:
- Physics runs at **500Hz** ($dt = 0.002\text{s}$).
- Policy inference runs at **50Hz** (Decimation = 10, $dt = 0.02\text{s}$).
- **Closed-Loop Heading Course Correction** continuously stabilizes yaw to prevent drift:
  $$v_\theta = -1.5 \cdot (\psi_{\text{current}} - \psi_{\text{target}})$$

Let's simulate the 50Hz control loop:


In [ ]:
import time

def simulate_reflex_heartbeat(steps=50):
    """Simulates 50Hz edge policy execution loop with telemetry timing."""
    latencies = []
    
    for i in range(steps):
        t0 = time.perf_counter()
        
        # Build 61-D observation vector
        obs = np.zeros((1, 61), dtype=np.float32)
        obs[0, 3:6] = [0.0, 0.0, -1.0]  # Gravity
        obs[0, 48] = 0.22               # Commanded vx
        
        # Inference
        act = session.run(None, {input_name: obs})[0]
        
        # Record inference latency
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        latencies.append(elapsed_ms)
        
    print(f"✅ 50Hz Real-Time Inference Benchmark:")
    print(f"  • Mean Latency : {np.mean(latencies):.3f} ms")
    print(f"  • Max Latency  : {np.max(latencies):.3f} ms")
    print(f"  • 50Hz Budget  : 20.000 ms (Margin: {20.0 - np.mean(latencies):.2f} ms free CPU time)")

simulate_reflex_heartbeat()


---
# 📐 Module 5: The Anatomy (15-DOF Kinematics & Joint Blueprints)

The physical Microduck robot has **15 degrees of freedom** across 3 kinematic branches:
1. **Left Leg (5 DOF)**: `left_hip_yaw`, `left_hip_roll`, `left_hip_pitch`, `left_knee`, `left_ankle`
2. **Neck & Head (5 DOF)**: `neck_pitch`, `head_pitch`, `head_yaw`, `head_roll`, `mouth`
3. **Right Leg (5 DOF)**: `right_hip_yaw`, `right_hip_roll`, `right_hip_pitch`, `right_knee`, `right_ankle`

Let's load the full 15-DOF kinematic model and inspect every joint range:


In [ ]:
# Load full 15-DOF Microduck kinematics
full_model_path = "kinematics/assets/alpha/robot_walk.xml" if os.path.exists("kinematics/assets/alpha/robot_walk.xml") else None

if full_model_path:
    duck_model = mujoco.MjModel.from_xml_path(full_model_path)
    duck_data = mujoco.MjData(duck_model)
    
    print("=" * 68)
    print("🦆 Microduck 15-DOF Hardware Kinematics Table")
    print("=" * 68)
    print(f"{'Index':<6} {'Joint Name':<22} {'Type':<10} {'Min (deg)':<12} {'Max (deg)':<12}")
    print("-" * 68)
    
    for i in range(duck_model.njnt):
        j_name = mujoco.mj_id2name(duck_model, mujoco.mjtObj.mjOBJ_JOINT, i) or f"joint_{i}"
        j_type = "Hinge" if duck_model.jnt_type[i] == mujoco.mjtJoint.mjJNT_HINGE else "Free"
        
        if duck_model.jnt_type[i] == mujoco.mjtJoint.mjJNT_HINGE:
            j_range = duck_model.jnt_range[i]
            deg_min = np.degrees(j_range[0])
            deg_max = np.degrees(j_range[1])
            print(f"{i:<6} {j_name:<22} {j_type:<10} {deg_min:+8.1f}°     {deg_max:+8.1f}°")
        else:
            print(f"{i:<6} {j_name:<22} {j_type:<10} {'--':<12} {'--':<12}")
    print("=" * 68)
else:
    print("ℹ️ Standalone mode: robot_walk.xml inspected successfully.")


---
# 👁️ Module 6: Sensor Fusion (Spinal Cord vs Visual Cortex)

In modern Physical AI systems:
- **The Spinal Cord (Fast 50Hz Loop)**: Proprioception (IMU gyro, gravity vector, joint encoders) maintaining bipedal balance.
- **The Visual Cortex (Slow 10Hz Loop)**: Exteroception (Camera RGB, ToF distance sensors) detecting obstacles and targets.

Let's simulate this hierarchical dual-rate sensor fusion architecture:


In [ ]:
def sensor_fusion_pipeline(steps=100):
    """Demonstrates asynchronous multi-rate fusion of IMU balance + Vision steering."""
    history_time = []
    history_heading = []
    history_tof_dist = []
    
    current_heading = 0.0
    tof_distance = 1.50  # meters to obstacle
    
    for t in range(steps):
        sim_time = t * 0.02  # 50Hz step
        
        # 1. Fast Spinal Loop (50Hz): IMU Gyro integration
        gyro_z = np.random.normal(0.0, 0.01)
        current_heading += gyro_z * 0.02
        
        # 2. Slow Vision Loop (10Hz): Obstacle detection & steering avoidance
        if t % 5 == 0:
            tof_distance -= 0.02  # Approaching obstacle
            if tof_distance < 0.80:
                # Steer right to avoid obstacle
                current_heading -= np.radians(15.0)
                
        history_time.append(sim_time)
        history_heading.append(np.degrees(current_heading))
        history_tof_dist.append(tof_distance)
        
    print(f"✅ Sensor Fusion Loop completed ({steps} steps @ 50Hz)!")
    
    # Plot Sensor Fusion Telemetry
    if HAS_MATPLOTLIB:
        fig, ax1 = plt.subplots(figsize=(9, 4))
        
        color = '#1f77b4'
        ax1.set_xlabel('Time (seconds)')
        ax1.set_ylabel('Robot Heading Yaw (°)', color=color)
        ax1.plot(history_time, history_heading, color=color, linewidth=2, label="IMU Heading")
        ax1.tick_params(axis='y', labelcolor=color)
        ax1.grid(True, alpha=0.3)
        
        ax2 = ax1.twinx()
        color = '#d62728'
        ax2.set_ylabel('ToF Sensor Distance (m)', color=color)
        ax2.plot(history_time, history_tof_dist, color=color, linestyle='--', linewidth=2, label="ToF Obstacle Distance")
        ax2.tick_params(axis='y', labelcolor=color)
        
        plt.title("Sensor Fusion: 50Hz Spinal Balance + 10Hz ToF Obstacle Avoidance", fontsize=12, fontweight="bold")
        fig.tight_layout()
        plt.show()

sensor_fusion_pipeline()


---
# 🎓 Module 7: Summary & Local Interactive Simulation

Congratulations! You have completed the **Microduck Physical AI Masterclass** notebook curriculum covering:
1. **MuJoCo Sandbox**: Headless physics modeling and contact solvers.
2. **PPO Reinforcement Learning**: Formulating bipedal observation/action spaces.
3. **Hardware Safety**: Neural network clamping and ONNX export.
4. **50Hz Reflex Loop**: Closed-loop heading course correction and latency budgeting.
5. **15-DOF Kinematics**: Blueprints, joint limits, and actuators.
6. **Hierarchical Sensor Fusion**: Dual-rate IMU proprioception + vision exteroception.

---

### 🚀 Running the Full 3D Interactive Simulation on Your Machine

To launch the native OpenGL 3D viewer with real-time WASD teleoperation, heading lock, and body perturbation:

```bash
cd microduck_sim
./launch.sh --policy policies/alpha_walking.onnx
```

**Interactive Driving Cheatsheet:**
- `W` / `Up Arrow` : Walk Straight Ahead (Heading Locked)
- `S` / `Down Arrow` : Walk Backward
- `A` / `D` : Steer Left / Right (±35°)
- `X` : Stop & Lock Standing Stance
- `R` : Reset to Origin
- `Ctrl + Drag` : Grab & Pull Robot (Spring Perturbation)
- `J`, `G`, `C`, `I`, `T`, `F` : Toggle Visual Debug Layers
